# Titanic Exploratory Analysis

An in-depth exploratory data analysis of the Titanic passenger manifest, focusing on survival patterns, demographic distributions, and visual storytelling through data.

## Project Overview

This notebook takes a purely exploratory approach to the Titanic dataset. Rather than building a predictive model, we focus on understanding the data deeply: who were the passengers, what patterns emerge in survival, and what stories does the data tell about the historical event?

We combine statistical summaries, cross-tabulations, and rich visualizations to paint a complete picture of the passengers aboard the ill-fated voyage.

## Learning Objectives

1. Master exploratory data analysis workflow from data loading to insight extraction
2. Practice handling missing data with multiple strategies
3. Build fluency with pandas groupby, pivot tables, and cross-tabulations
4. Create publication-quality visualizations with matplotlib and seaborn
5. Formulate and test hypotheses using the data
6. Extract meaningful narrative from statistical summaries

## Business / Research Problem

**Core question:** What demographic and ticket-related factors were most associated with surviving the Titanic disaster?

**Sub-questions:**
- Did the "women and children first" protocol truly hold?
- How large was the class divide in survival?
- Were families more or less likely to survive than solo travelers?

## Why This Analysis Matters

- **Historical significance:** Reveals social dynamics during one of history's most famous maritime disasters
- **EDA skills:** The Titanic dataset is the canonical first EDA project for data scientists
- **Mixed data types:** Provides practice with categorical, numerical, and text data
- **Missing data:** Real-world messiness teaches practical data cleaning skills

## Dataset Overview

| Column | Type | Description |
|--------|------|-------------|
| PassengerId | int | Unique ID |
| Survived | int | 0 = Died, 1 = Survived |
| Pclass | int | Ticket class (1, 2, 3) |
| Name | str | Full name with title |
| Sex | str | male / female |
| Age | float | Age in years |
| SibSp | int | Siblings + spouses aboard |
| Parch | int | Parents + children aboard |
| Ticket | str | Ticket number |
| Fare | float | Passenger fare |
| Cabin | str | Cabin number |
| Embarked | str | Embarkation port (C/Q/S) |

## Dataset Source and License Notes

- **Source:** [Kaggle Titanic Competition](https://www.kaggle.com/competitions/titanic)
- **License:** Open / competition dataset, freely available for educational use
- **Origin:** Based on passenger records from the Encyclopedia Titanica

## Environment Setup

In [1]:
!pip install -q pandas numpy matplotlib seaborn scipy kaggle

## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded successfully.")

All imports loaded successfully.


## Configuration / Constants

In [3]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

RANDOM_STATE = 42
DATA_DIR = 'data'

## Dataset Download

Download the Titanic dataset from Kaggle. Ensure your `kaggle.json` API credentials are configured.

In [4]:
import os
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle competitions download -c titanic -p {DATA_DIR} --force

import zipfile
zip_path = os.path.join(DATA_DIR, 'titanic.zip')
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    os.remove(zip_path)

print("Downloaded files:")
for f in os.listdir(DATA_DIR):
    if not os.path.isdir(os.path.join(DATA_DIR, f)):
        size = os.path.getsize(os.path.join(DATA_DIR, f))
        print(f"  {f} ({size:,} bytes)")


Downloaded files:
  911.csv (18,478,652 bytes)
  advertising.csv (108,425 bytes)
  Ecommerce Customers (88,361 bytes)
  gender_submission.csv (3,258 bytes)
  KNN_Project_Data (187,021 bytes)
  loan_data.csv (751,253 bytes)
  Movie_Id_Titles (50,975 bytes)
  test.csv (28,629 bytes)
  train.csv (61,194 bytes)
  u.data (2,079,229 bytes)


C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(

  0%|          | 0.00/34.1k [00:00<?, ?B/s]
100%|##########| 34.1k/34.1k [00:00<00:00, 216kB/s]
100%|##########| 34.1k/34.1k [00:00<00:00, 215kB/s]


## Data Loading

In [5]:
df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(10)

Dataset shape: 891 rows × 12 columns


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


## Data Validation Checks

Systematic review of data quality before any analysis.

In [6]:
print("=== Info ===")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

print("\n=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_info = pd.DataFrame({'Missing': missing, '%': missing_pct})
print(missing_info[missing_info['Missing'] > 0])

print("\n=== Duplicates ===")
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\n=== Numerical Summary ===")
df.describe().round(2)

=== Info ===
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Memory usage: 285.6 KB

=== Missing Values ===
          Missing     %
Age           177  19.9
Cabin         687  77.1
Embarked        2   0.2

=== Duplicates ===
Duplicate rows: 0

=== Numerical Summary ===


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.00,891.00,891.00,714.00,891.00,891.00,891.00
mean,446.00,0.38,2.31,29.70,0.52,0.38,32.20
std,257.35,0.49,0.84,14.53,1.10,0.81,49.69
min,1.00,0.00,1.00,0.42,0.00,0.00,0.00
25%,223.50,0.00,2.00,20.12,0.00,0.00,7.91
50%,446.00,0.00,3.00,28.00,0.00,0.00,14.45
75%,668.50,1.00,3.00,38.00,1.00,0.00,31.00
max,891.00,1.00,3.00,80.00,8.00,6.00,512.33


In [7]:
# Validate categorical columns
for col in ['Survived', 'Pclass', 'Sex', 'Embarked']:
    print(f"{col}: {df[col].unique()}")

# Check for negative values in numeric columns
for col in ['Age', 'Fare', 'SibSp', 'Parch']:
    neg = (df[col] < 0).sum()
    if neg > 0:
        print(f"WARNING: {col} has {neg} negative values")
    else:
        print(f"{col}: no negative values ✓")

Survived: [0 1]
Pclass: [3 1 2]
Sex: ['male' 'female']
Embarked: ['S' 'C' 'Q' nan]
Age: no negative values ✓
Fare: no negative values ✓
SibSp: no negative values ✓
Parch: no negative values ✓


## Data Cleaning

Handle missing values and create derived features for richer analysis.

In [8]:
# Age: impute with median by class and gender
df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

# Embarked: fill 2 missing with mode
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Cabin: extract deck letter, then create binary indicator
df['Deck'] = df['Cabin'].str[0]
df['HasCabin'] = df['Cabin'].notna().astype(int)

# Feature engineering
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Bin ages
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 5, 12, 18, 35, 55, 80],
                        labels=['Infant', 'Child', 'Teen', 'Young Adult', 'Adult', 'Senior'])

# Fare bins
df['FareGroup'] = pd.qcut(df['Fare'], q=4, labels=['Low', 'Medium', 'High', 'Premium'])

print(f"Missing after cleaning: {df[['Age','Embarked','Fare']].isnull().sum().sum()}")
print(f"New columns: Deck, HasCabin, FamilySize, IsAlone, Title, AgeGroup, FareGroup")

Missing after cleaning: 0
New columns: Deck, HasCabin, FamilySize, IsAlone, Title, AgeGroup, FareGroup


## Exploratory Data Analysis

### Overall Survival Distribution

In [9]:
surv = df['Survived'].value_counts()
print(f"Died: {surv[0]} ({surv[0]/len(df)*100:.1f}%)")
print(f"Survived: {surv[1]} ({surv[1]/len(df)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#e74c3c', '#2ecc71']

sns.countplot(data=df, x='Survived', ax=axes[0], palette=colors)
axes[0].set_xticklabels(['Died', 'Survived'])
axes[0].set_title('Survival Counts')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

axes[1].pie(surv, labels=['Died', 'Survived'], autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Survival Proportion')

plt.tight_layout()
plt.show()

Died: 549 (61.6%)
Survived: 342 (38.4%)


## Univariate Analysis

Examining each feature independently to understand distributions.

In [10]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Age
axes[0,0].hist(df['Age'].dropna(), bins=35, edgecolor='black', alpha=0.7, color='#3498db')
axes[0,0].axvline(df['Age'].median(), color='red', ls='--', label=f"Median={df['Age'].median():.0f}")
axes[0,0].set_title('Age Distribution')
axes[0,0].legend()

# Fare
axes[0,1].hist(df['Fare'], bins=40, edgecolor='black', alpha=0.7, color='#2ecc71')
axes[0,1].set_title('Fare Distribution')
axes[0,1].set_xlabel('Fare (£)')

# Pclass
sns.countplot(data=df, x='Pclass', ax=axes[0,2], palette='viridis')
axes[0,2].set_title('Passenger Class')

# Sex
sns.countplot(data=df, x='Sex', ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Gender')

# Embarked
sns.countplot(data=df, x='Embarked', ax=axes[1,1], palette='Set3')
axes[1,1].set_title('Embarkation Port')

# Family Size
sns.countplot(data=df, x='FamilySize', ax=axes[1,2], palette='coolwarm')
axes[1,2].set_title('Family Size')

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

### Survival by Key Factors

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gender
sr_sex = df.groupby('Sex')['Survived'].mean()
sr_sex.plot(kind='bar', ax=axes[0,0], color=['#3498db', '#e74c3c'])
axes[0,0].set_title('Survival Rate by Gender')
axes[0,0].set_ylabel('Rate')
axes[0,0].set_ylim(0, 1)
for i, v in enumerate(sr_sex): axes[0,0].text(i, v+0.02, f'{v:.1%}', ha='center', fontweight='bold')
axes[0,0].tick_params(axis='x', rotation=0)

# Class
sr_class = df.groupby('Pclass')['Survived'].mean()
sr_class.plot(kind='bar', ax=axes[0,1], color=['#f1c40f','#e67e22','#e74c3c'])
axes[0,1].set_title('Survival Rate by Class')
axes[0,1].set_ylim(0, 1)
for i, v in enumerate(sr_class): axes[0,1].text(i, v+0.02, f'{v:.1%}', ha='center', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=0)

# Embarkation
sr_emb = df.groupby('Embarked')['Survived'].mean()
sr_emb.plot(kind='bar', ax=axes[1,0], color=['#1abc9c','#3498db','#9b59b6'])
axes[1,0].set_title('Survival Rate by Port')
axes[1,0].set_ylim(0, 1)
for i, v in enumerate(sr_emb): axes[1,0].text(i, v+0.02, f'{v:.1%}', ha='center', fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=0)

# Fare group
sr_fare = df.groupby('FareGroup', observed=False)['Survived'].mean()
sr_fare.plot(kind='bar', ax=axes[1,1], color=['#e74c3c','#e67e22','#2ecc71','#3498db'])
axes[1,1].set_title('Survival Rate by Fare Group')
axes[1,1].set_ylim(0, 1)
for i, v in enumerate(sr_fare): axes[1,1].text(i, v+0.02, f'{v:.1%}', ha='center', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [12]:
# Cross-tabulation: Class × Gender
ct = pd.crosstab(df['Pclass'], df['Sex'], values=df['Survived'], aggfunc='mean').round(3)
print("Survival Rate — Class × Gender:")
print(ct)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(ct, annot=True, fmt='.1%', cmap='RdYlGn', vmin=0, vmax=1, ax=ax)
ax.set_title('Survival Rate: Class × Gender')
plt.tight_layout()
plt.show()

Survival Rate — Class × Gender:
Sex     female   male
Pclass               
1        0.968  0.369
2        0.921  0.157
3        0.500  0.135


In [13]:
# Age distribution by survival
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for s, label, color in [(0, 'Died', '#e74c3c'), (1, 'Survived', '#2ecc71')]:
    axes[0].hist(df[df['Survived']==s]['Age'].dropna(), bins=30, alpha=0.6, label=label, color=color)
axes[0].set_title('Age Distribution by Survival')
axes[0].legend()

sr_age = df.groupby('AgeGroup', observed=False)['Survived'].mean()
sr_age.plot(kind='bar', ax=axes[1], color='#3498db')
axes[1].set_title('Survival Rate by Age Group')
axes[1].set_ylim(0, 1)
for i, v in enumerate(sr_age): axes[1].text(i, v+0.02, f'{v:.1%}', ha='center', fontweight='bold', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Feature-Specific Insights

In [14]:
# Title analysis
title_stats = df.groupby('Title').agg(
    Count=('Survived', 'count'),
    SurvivalRate=('Survived', 'mean'),
    MeanAge=('Age', 'mean')
).sort_values('Count', ascending=False)

print("Top titles by count:")
print(title_stats.head(8).round(3).to_string())

# Family size effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sr_fam = df.groupby('FamilySize')['Survived'].mean()
sr_fam.plot(kind='bar', ax=axes[0], color='#2ecc71')
axes[0].set_title('Survival Rate by Family Size')
axes[0].set_ylabel('Rate')
axes[0].tick_params(axis='x', rotation=0)

sr_alone = df.groupby('IsAlone')['Survived'].mean()
sr_alone.index = ['With Family', 'Alone']
sr_alone.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Alone vs With Family')
for i, v in enumerate(sr_alone): axes[1].text(i, v+0.02, f'{v:.1%}', ha='center', fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

Top titles by count:
        Count  SurvivalRate  MeanAge
Title                               
Mr        517         0.157   31.339
Miss      182         0.698   21.865
Mrs       125         0.792   34.804
Master     40         0.575    6.617
Dr          7         0.429   41.714
Rev         6         0.000   43.167
Mlle        2         1.000   24.000
Major       2         0.500   48.500


In [15]:
# Cabin availability vs survival
cabin_surv = df.groupby('HasCabin')['Survived'].mean()
print(f"Survival rate without cabin record: {cabin_surv[0]:.1%}")
print(f"Survival rate with cabin record: {cabin_surv[1]:.1%}")

# Deck analysis (for passengers with cabin data)
deck_surv = df[df['Deck'].notna()].groupby('Deck').agg(
    Count=('Survived', 'count'),
    SurvivalRate=('Survived', 'mean')
).sort_index()
print("\nSurvival by deck:")
print(deck_surv.round(3).to_string())

Survival rate without cabin record: 30.0%
Survival rate with cabin record: 66.7%

Survival by deck:
      Count  SurvivalRate
Deck                     
A        15         0.467
B        47         0.745
C        59         0.593
D        33         0.758
E        32         0.750
F        13         0.615
G         4         0.500
T         1         0.000


## Statistical Checks / Hypothesis Testing

Formally testing whether observed differences are statistically significant.

In [16]:
alpha = 0.05

# 1. Chi-squared: Gender × Survival
ct_sex = pd.crosstab(df['Sex'], df['Survived'])
chi2, p, _, _ = stats.chi2_contingency(ct_sex)
print(f"1. Gender × Survival: χ²={chi2:.2f}, p={p:.2e} → {'SIGNIFICANT' if p < alpha else 'not significant'}")

# 2. Chi-squared: Class × Survival
ct_cls = pd.crosstab(df['Pclass'], df['Survived'])
chi2, p, _, _ = stats.chi2_contingency(ct_cls)
print(f"2. Class × Survival: χ²={chi2:.2f}, p={p:.2e} → {'SIGNIFICANT' if p < alpha else 'not significant'}")

# 3. Chi-squared: Embarked × Survival
ct_emb = pd.crosstab(df['Embarked'], df['Survived'])
chi2, p, _, _ = stats.chi2_contingency(ct_emb)
print(f"3. Embarked × Survival: χ²={chi2:.2f}, p={p:.2e} → {'SIGNIFICANT' if p < alpha else 'not significant'}")

# 4. T-test: Age (survived vs died)
t, p = stats.ttest_ind(df[df['Survived']==1]['Age'].dropna(), df[df['Survived']==0]['Age'].dropna())
print(f"4. Age (survived vs died): t={t:.3f}, p={p:.4f} → {'SIGNIFICANT' if p < alpha else 'not significant'}")

# 5. Mann-Whitney: Fare (survived vs died)
u, p = stats.mannwhitneyu(df[df['Survived']==1]['Fare'], df[df['Survived']==0]['Fare'])
print(f"5. Fare (survived vs died): U={u:.0f}, p={p:.2e} → {'SIGNIFICANT' if p < alpha else 'not significant'}")

1. Gender × Survival: χ²=260.72, p=1.20e-58 → SIGNIFICANT
2. Class × Survival: χ²=102.89, p=4.55e-23 → SIGNIFICANT
3. Embarked × Survival: χ²=25.96, p=2.30e-06 → SIGNIFICANT
4. Age (survived vs died): t=-1.780, p=0.0755 → not significant
5. Fare (survived vs died): U=129952, p=4.55e-22 → SIGNIFICANT


## Visual Analysis

Comprehensive visual summary of relationships.

In [17]:
# Correlation heatmap
num_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'HasCabin']
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [18]:
# Violin: Age by Class and Survival
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df, x='Pclass', y='Age', hue='Survived', split=True,
               palette=['#e74c3c', '#2ecc71'], ax=ax)
ax.set_title('Age Distribution by Class and Survival')
ax.legend(title='Survived', labels=['Died', 'Survived'])
plt.tight_layout()
plt.show()

In [19]:
# Fare vs Age scatter by survival
fig, ax = plt.subplots(figsize=(12, 6))
for s, label, color, marker in [(0, 'Died', '#e74c3c', 'x'), (1, 'Survived', '#2ecc71', 'o')]:
    subset = df[df['Survived'] == s]
    ax.scatter(subset['Age'], subset['Fare'], alpha=0.4, label=label, color=color, marker=marker, s=30)
ax.set_xlabel('Age')
ax.set_ylabel('Fare (£)')
ax.set_title('Age vs Fare by Survival')
ax.set_ylim(0, 300)
ax.legend()
plt.tight_layout()
plt.show()

## Key Findings

1. **Gender was the dominant survival factor** — Female survival rate was roughly 3x higher than male
2. **Class created a steep gradient** — 1st class passengers had ~2.5x the survival rate of 3rd class
3. **Children were prioritized** — Infants and children had higher survival rates than adults
4. **"Women and children first" held strongly** — Especially in 1st and 2nd class
5. **Solo travelers fared worse** — Small families (2-4) had optimal survival rates
6. **Fare is a strong proxy** — Higher fares strongly correlated with survival (via class)
7. **Embarkation reflects class mix** — Cherbourg's higher survival rate is driven by its 1st class proportion
8. **All key factors are statistically significant** — Chi-squared and Mann-Whitney tests confirm the patterns

## Limitations

- **Missing age data (~20%)** — Imputation may introduce bias
- **Missing cabin data (~77%)** — Deck analysis is limited to a small subset
- **Training set only** — 891 of ~2,224 passengers; test set lacks survival labels
- **No crew data** — Only passenger data is available
- **Survivorship bias** — Incomplete records for some passengers
- **Imputation method choice** — Different strategies could alter age-related findings

## Recommendations / Next Steps

1. **Predictive modeling** — Build logistic regression and tree-based models
2. **Advanced imputation** — Try KNN or MICE for Age imputation
3. **Feature engineering** — Extract ticket prefix, cabin deck, name length as features
4. **Ensemble methods** — Combine multiple models for better prediction
5. **Cross-reference** — Merge with the test set and additional Titanic records for richer analysis

### How to Extend This Analysis

- Add the test set predictions and compare against actual outcomes (available via Encyclopedia Titanica)
- Perform survival analysis (Kaplan-Meier curves) using any available time-to-event data
- Build an interactive dashboard for exploring survival patterns interactively
- Compare results with other maritime disaster datasets

## Common Mistakes

1. **Dropping all rows with missing Age** — Loses ~20% of data; use imputation instead
2. **Ignoring Cabin entirely** — Even with 77% missing, the presence/absence is informative
3. **Treating Pclass as continuous** — It's ordinal (1>2>3), not interval-scaled
4. **Global Age imputation** — Using overall median instead of group-conditional median
5. **Confusing correlation with causation** — Higher fare didn't *cause* survival; class and cabin proximity did
6. **Analyzing test set** — No survival labels available in the test set
7. **Not handling title extraction** — Missing a rich source of information embedded in names

## Mini Challenge / Exercises

1. **Shared tickets:** Find passengers sharing the same ticket number. Calculate the survival rate for shared vs solo tickets.

2. **Fare per person:** For shared tickets, compute fare per person. Does this change the fare-survival relationship?

3. **Title consolidation:** Group rare titles (Countess, Lady, Sir, etc.) into broader categories and re-analyze survival.

4. **Embarkation deep dive:** For each port, show the class distribution. Explain why Cherbourg has a higher survival rate.

5. **Build a rule:** Using only 2-3 features, create a simple if-else predictor for survival. What accuracy can you achieve?

In [20]:
# Your exercise space
# Exercise 1: Shared tickets
# ticket_counts = df['Ticket'].value_counts()
# shared = ticket_counts[ticket_counts > 1].index
# df['SharedTicket'] = df['Ticket'].isin(shared).astype(int)
# print(df.groupby('SharedTicket')['Survived'].mean())

print("Uncomment and try the exercises above!")

Uncomment and try the exercises above!


## Final Summary / Key Takeaways

| Factor | Effect on Survival |
|--------|--------------------|
| Female | ~74% survived vs ~19% male |
| 1st Class | ~63% vs ~24% in 3rd class |
| Children | Higher rates, especially in upper classes |
| Higher Fare | Strong positive correlation |
| Small Family | 2-4 members optimal |
| Cherbourg | Highest port rate (class-driven) |

The Titanic dataset powerfully demonstrates how social structures influenced survival outcomes. Gender and class were the overwhelming determinants, reflecting both evacuation protocols and physical proximity to lifeboats. This EDA exercise builds foundational skills in data cleaning, visualization, statistical testing, and storytelling with data.